# Modeling

## Imports

In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, learning_curve
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

import matplotlib.pyplot as plt

import os
import sys
import numpy as np

from dotenv import load_dotenv

# Cargar las del archivo .env (si existe)
load_dotenv()

True

In [6]:
# Custom modules
# Obtener el directorio del notebook (notebooks/)
notebook_dir = os.getcwd()
# Subir un nivel para llegar a la raíz del proyecto
project_root = os.path.abspath(os.path.join(notebook_dir, '..'))
# Agregar al path si no está (solo afecta a la instancia actual)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.data_management import get_categorical_number_columns
from src.model_trainer import ModelTrainer

## Cargar Dataset

In [ ]:
file_traintest = os.path.join('..','data', 'clean',os.getenv('FILE_TRAINTEST'))
df = pd.read_csv(file_traintest)

## Separacion de variables

In [8]:
# Variable objetivo (target)
y = df["MonthlyIncome"]

# Variables explicativas
X = df.drop(columns=["MonthlyIncome"])

print("\nShape de X:", X.shape)
print("Shape de y:", y.shape)


Shape de X: (1176, 14)
Shape de y: (1176,)


## Definir variables Categoricas y numericas

In [9]:
# Convertir columnas numéricas categóricas a tipo 'object'
categoricas_numericas = get_categorical_number_columns()

for col in categoricas_numericas:
    if col in X.columns:
        X[col] = X[col].astype("object")


# Obtener listas de variables categóricas y numéricas
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print("\nVariables categóricas:")
print(cat_cols)

print("\nVariables numéricas:")
print(num_cols)


Variables categóricas:
['Attrition', 'BusinessTravel', 'Department', 'Education', 'EducationField', 'JobInvolvement', 'JobRole', 'JobSatisfaction', 'OverTime', 'PerformanceRating', 'WorkLifeBalance']

Variables numéricas:
['JobLevel', 'YearsAtCompany', 'YearsSinceLastPromotion']


## Definición del preprocesador

In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ("num", StandardScaler(), num_cols)
    ]
)


## Separación en Train y Test

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("\nShape X_train:", X_train.shape)
print("Shape X_test:", X_test.shape)
print("Shape y_train:", y_train.shape)
print("Shape y_test:", y_test.shape)



Shape X_train: (940, 14)
Shape X_test: (236, 14)
Shape y_train: (940,)
Shape y_test: (236,)


## Definicion modelos a evaluar

In [12]:
modelos = {
    "RegresionLineal": LinearRegression(),
    "RandomForest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='reg:squarederror',
        random_state=42
    )
}

## Entrenamiento y Evaluacion de Modelos

In [13]:
# Crear trainer
figures = os.path.join('..','outputs','figures')
models = os.path.join('..','models')
trainer = ModelTrainer(modelos, preprocessor, figures_dir=figures, models_dir=models)

# Entrenar todos los modelos
trainer.entrenar_todos(X_train, y_train, X_test, y_test, 
                       plot_predictions=True, 
                       plot_learning=True)

# Guardar mejor modelo
trainer.guardar_mejor_modelo()

# Obtener resultados como DataFrame
resultados_df = trainer.obtener_resultados()


Entrenando: RegresionLineal
  → Entrenando modelo...
  → Métricas:
     MAE  = 879.42
     RMSE = 1142.00
     R²   = 0.9343
     MAPE = 19.52%
  ✓ Nuevo mejor modelo!
  → Gráfico guardado: ../outputs/figures/real_vs_predicho_RegresionLineal.png
  → Generando curva de aprendizaje...
  → Curva guardada: ../outputs/figures/learning_curve_RegresionLineal.png

Entrenando: RandomForest
  → Entrenando modelo...
  → Métricas:
     MAE  = 820.62
     RMSE = 1072.56
     R²   = 0.9420
     MAPE = 18.87%
  ✓ Nuevo mejor modelo!
  → Gráfico guardado: ../outputs/figures/real_vs_predicho_RandomForest.png
  → Generando curva de aprendizaje...
  → Curva guardada: ../outputs/figures/learning_curve_RandomForest.png

Entrenando: XGBoost
  → Entrenando modelo...
  → Métricas:
     MAE  = 838.67
     RMSE = 1087.71
     R²   = 0.9404
     MAPE = 19.28%
  → Gráfico guardado: ../outputs/figures/real_vs_predicho_XGBoost.png
  → Generando curva de aprendizaje...
  → Curva guardada: ../outputs/figures/learnin